In [ ]:
import os

html_content = """<!DOCTYPE html>
<html>
    <head>
        <title>Test HTML</title>
    </head>
    <body>
        <h1>Welcome to HTML Compiler</h1>
        <p>This is a <b>simple</b> example.</p>
        <div>
            <h2>Unclosed Tags Test
            <p>Another paragraph without closing.</div>
        <br>
    </body>
</html>"""

os.makedirs("input", exist_ok=True)  # Create the directory if it doesn't exist
with open("input/input.html", "w") as f:
  f.write(html_content)

print("HTML file created successfully at input/input.html")


HTML file created successfully at input/input.html


In [ ]:
from html.parser import HTMLParser
import webbrowser
import os
import time
import re
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("html_compiler.log"), logging.StreamHandler()]
)

class HTMLCompiler(HTMLParser):
    """
    An HTML parser that validates, corrects, and beautifies HTML code.
    Features:
    - Tag structure validation
    - Required attribute checking
    - CSS validation
    - Self-closing tag handling
    - Proper indentation in output
    - Accessibility recommendations
    - Performance suggestions
    """

    def __init__(self):
        super().__init__()

        self.stack = []
        self.errors = []
        self.warnings = []
        self.suggestions = []
        self.ids = {}
        self.classes = {}
        self.inline_styles = []
        self.accessibility_issues = []


        self.tag_count = {}
        self.total_size = 0


        self.corrected_html = []
        self.raw_corrected_html = []
        self.indent_level = 0
        self.line_buffer = ""


        self.self_closing_tags = {
            'area', 'base', 'br', 'col', 'embed', 'hr', 'img',
            'input', 'link', 'meta', 'param', 'source', 'track', 'wbr'
        }


        self.required_attributes = {
            'img': ['src', 'alt'],
            'a': ['href'],
            'link': ['href'],
            'input': ['type'],
            'meta': ['content'],
            'script': ['src'],
            'form': ['action']
        }


        self.valid_parent_child = {
            'ul': ['li'],
            'ol': ['li'],
            'table': ['thead', 'tbody', 'tfoot', 'tr'],
            'tr': ['th', 'td'],
            'select': ['option', 'optgroup'],
            'dl': ['dt', 'dd']
        }


        self.raw_corrected_html.append("<!DOCTYPE html>")
        self.raw_corrected_html.append('<html lang="en">')


        self._initialize_output()

    def _initialize_output(self):
        """Initialize the HTML output with doctype and basic structure"""
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        self.corrected_html = [
            "<!DOCTYPE html>",
            '<html lang="en">',
            '<head>',
            '    <meta charset="UTF-8">',
            '    <meta name="viewport" content="width=device-width, initial-scale=1.0">',
            f'    <title>HTML Compiler Output - {current_time}</title>',
            '    <style>',
            '        body { font-family: "Segoe UI", Arial, sans-serif; line-height: 1.6; color: #333; max-width: 1200px; margin: 0 auto; padding: 20px; }',
            '        .container { display: grid; grid-template-columns: 1fr 300px; gap: 20px; }',
            '        .content { border: 1px solid #ddd; padding: 20px; border-radius: 5px; }',
            '        .sidebar { background: #f8f9fa; padding: 15px; border-radius: 5px; }',
            '        h1, h2, h3 { color: #2c3e50; }',
            '        .error { color: #e74c3c; font-weight: bold; }',
            '        .warning { color: #f39c12; }',
            '        .success { color: #27ae60; font-weight: bold; }',
            '        .suggestion { color: #3498db; }',
            '        pre { background: #f8f9fa; padding: 10px; border-radius: 5px; overflow-x: auto; border: 1px solid #ddd; }',
            '        .stats { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }',
            '        .stat-card { background: #fff; border: 1px solid #ddd; padding: 10px; border-radius: 5px; }',
            '        .footer { margin-top: 20px; text-align: center; font-size: 0.9em; color: #7f8c8d; }',
            '        .code-display { font-family: monospace; white-space: pre-wrap; }',
            '        .tabs { display: flex; border-bottom: 1px solid #ddd; }',
            '        .tab { padding: 10px 15px; cursor: pointer; }',
            '        .tab.active { border-bottom: 2px solid #3498db; color: #3498db; }',
            '        .tab-content { display: none; }',
            '        .tab-content.active { display: block; }',
            '    </style>',
            '</head>',
            '<body>',
            '    <h1>HTML Compiler Results</h1>',
            '    <div class="tabs">',
            '        <div class="tab active" onclick="switchTab(\'preview\')">Preview</div>',
            '        <div class="tab" onclick="switchTab(\'corrected-code\')">Corrected Code</div>',
            '        <div class="tab" onclick="switchTab(\'original-code\')">Original Code</div>',
            '    </div>',
            '    <div class="container">',
            '        <div id="preview" class="content tab-content active">',
            '            <h2>Rendered Output</h2>',
            '            <div id="output-preview">'
        ]


        self.indent_level = 4

    def handle_starttag(self, tag, attrs):
        """Handle the start of an HTML tag"""

        self.tag_count[tag] = self.tag_count.get(tag, 0) + 1

        attr_dict = dict(attrs)

        if "id" in attr_dict:
            if attr_dict["id"] in self.ids:
                self.errors.append(f"Duplicate ID '{attr_dict['id']}' found in <{tag}> element.")
                attr_dict["id"] = f"{attr_dict['id']}-{len(self.ids)}"
                attrs = [(k, v) if k != "id" else ("id", attr_dict["id"]) for k, v in attrs]
            else:
                self.ids[attr_dict["id"]] = tag


        if "class" in attr_dict:
            classes = attr_dict["class"].split()
            for cls in classes:
                if cls not in self.classes:
                    self.classes[cls] = []
                self.classes[cls].append(tag)


        if "style" in attr_dict:
            self.inline_styles.append((tag, attr_dict["style"]))
            self.suggestions.append(f"Consider moving inline style '{attr_dict['style']}' from <{tag}> to an external stylesheet.")


        if tag in self.required_attributes:
            for required_attr in self.required_attributes[tag]:
                if required_attr not in attr_dict:
                    self.errors.append(f"<{tag}> tag missing required '{required_attr}' attribute.")

                    if required_attr == "src":
                        attrs.append((required_attr, "placeholder.jpg"))
                    elif required_attr == "href":
                        attrs.append((required_attr, "#"))
                    elif required_attr == "alt":
                        attrs.append((required_attr, "Image description"))
                    else:
                        attrs.append((required_attr, ""))


        if tag == "img" and ("alt" not in attr_dict or not attr_dict["alt"]):
            self.accessibility_issues.append("Images should have descriptive alt text for screen readers.")

        if tag == "a" and "title" not in attr_dict:
            self.warnings.append(f"Anchor tag missing 'title' attribute for better accessibility.")


        if self.stack and self.stack[-1] in self.valid_parent_child:
            valid_children = self.valid_parent_child[self.stack[-1]]
            if tag not in valid_children:
                self.warnings.append(f"<{tag}> may not be a valid direct child of <{self.stack[-1]}>.")


        if tag not in self.self_closing_tags:
            self.stack.append(tag)


        attr_str = " ".join(f'{key}="{value}"' for key, value in attrs)
        attr_str = f" {attr_str}" if attr_str else ""


        self._append_with_indent(f"<{tag}{attr_str}>")


        raw_indent = '    ' * len(self.stack) if tag not in self.self_closing_tags else '    ' * (len(self.stack) + 1)
        self.raw_corrected_html.append(f"{raw_indent}<{tag}{attr_str}>")


        if tag not in self.self_closing_tags and tag not in ['br', 'hr']:
            self.indent_level += 1

    def handle_endtag(self, tag):


        if self.stack and tag == self.stack[-1]:
            self.stack.pop()
        else:
            if tag not in self.self_closing_tags:
                self.errors.append(f"Unmatched closing tag </{ tag }> or tags closed in wrong order.")


        if tag not in self.self_closing_tags:
            self.indent_level -= 1
            self._append_with_indent(f"</{tag}>")


            raw_indent = '    ' * len(self.stack)
            self.raw_corrected_html.append(f"{raw_indent}</{tag}>")

    def handle_data(self, data):
        """Handle text data between tags"""

        if data.strip():

            formatted_data = re.sub(r'\s+', ' ', data.strip())
            self._append_with_indent(formatted_data)


            raw_indent = '    ' * len(self.stack)
            self.raw_corrected_html.append(f"{raw_indent}{formatted_data}")

    def handle_comment(self, data):
        """Handle HTML comments"""
        self._append_with_indent(f"<!-- {data} -->")


        raw_indent = '    ' * len(self.stack)
        self.raw_corrected_html.append(f"{raw_indent}<!-- {data} -->")

    def handle_decl(self, decl):
        """Handle doctype declarations"""
        self._append_with_indent(f"<!{decl}>")



    def _append_with_indent(self, content):
        """Append content to the corrected HTML with proper indentation"""

        indentation = '    ' * self.indent_level
        self.corrected_html.append(f"{indentation}{content}")

    def error_check(self):
        """Check for any unclosed tags and finalize the correction"""

        while self.stack:
            tag = self.stack.pop()
            self.errors.append(f"Unclosed <{tag}> tag.")

            self.indent_level -= 1
            self._append_with_indent(f"</{tag}>")


            raw_indent = '    ' * len(self.stack)
            self.raw_corrected_html.append(f"{raw_indent}</{tag}>")


        if not any(tag == 'html' for tag in self.raw_corrected_html):
            self.raw_corrected_html.append('</html>')


        self.corrected_html.extend([
            '            </div>',
            '        </div>',
            '        <div id="corrected-code" class="content tab-content">',
            '            <h2>Corrected HTML Code</h2>',
            '            <pre class="code-display" id="corrected-code-display"></pre>',
            '        </div>',
            '        <div id="original-code" class="content tab-content">',
            '            <h2>Original HTML Code</h2>',
            '            <pre class="code-display" id="original-code-display"></pre>',
            '        </div>',
            '        <div class="sidebar">',
            '            <h2>Diagnostics</h2>'
        ])

        if self.errors:
            self.corrected_html.extend([
                '            <h3 class="error">Errors Found:</h3>',
                '            <ul class="error">'
            ])
            for error in self.errors:
                self.corrected_html.append(f'                <li>{error}</li>')
            self.corrected_html.append('            </ul>')
        else:
            self.corrected_html.append('            <p class="success">No Errors Found!</p>')


        if self.warnings:
            self.corrected_html.extend([
                '            <h3 class="warning">Warnings:</h3>',
                '            <ul class="warning">'
            ])
            for warning in self.warnings:
                self.corrected_html.append(f'                <li>{warning}</li>')
            self.corrected_html.append('            </ul>')

        if self.accessibility_issues:
            self.corrected_html.extend([
                '            <h3>Accessibility Issues:</h3>',
                '            <ul class="warning">'
            ])
            for issue in self.accessibility_issues:
                self.corrected_html.append(f'                <li>{issue}</li>')
            self.corrected_html.append('            </ul>')


        if self.suggestions:
            self.corrected_html.extend([
                '            <h3 class="suggestion">Suggestions:</h3>',
                '            <ul class="suggestion">'
            ])
            for suggestion in self.suggestions:
                self.corrected_html.append(f'                <li>{suggestion}</li>')
            self.corrected_html.append('            </ul>')


        self.corrected_html.extend([
            '            <h3>Statistics:</h3>',
            '            <div class="stats">'
        ])

        self.corrected_html.append('                <div class="stat-card">')
        self.corrected_html.append('                    <h4>Tag Usage:</h4>')
        self.corrected_html.append('                    <ul>')
        for tag, count in sorted(self.tag_count.items(), key=lambda x: x[1], reverse=True)[:5]:
            self.corrected_html.append(f'                        <li><code>{tag}</code>: {count}</li>')
        self.corrected_html.append('                    </ul>')
        self.corrected_html.append('                </div>')

        # Overall statistics
        self.corrected_html.extend([
            '                <div class="stat-card">',
            '                    <h4>Summary:</h4>',
            f'                    <p>Total Tags: {sum(self.tag_count.values())}</p>',
            f'                    <p>Unique Tags: {len(self.tag_count)}</p>',
            f'                    <p>Total Errors: {len(self.errors)}</p>',
            f'                    <p>Total Warnings: {len(self.warnings)}</p>',
            '                </div>',
            '            </div>'
        ])

        # Close sidebar and container
        self.corrected_html.extend([
            '        </div>',
            '    </div>'
        ])

        # footer
        self.corrected_html.extend([
            '    <script>',
            '        // Function to escape HTML for display',
            '        function escapeHtml(html) {',
            '            return html.replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;")',
            '                       .replace(/"/g, "&quot;").replace(/\'/g, "&#039;");',
            '        }',
            '        ',
            '        // Function to switch between tabs',
            '        function switchTab(tabId) {',
            '            // Hide all tab contents',
            '            document.querySelectorAll(".tab-content").forEach(tab => {',
            '                tab.classList.remove("active");',
            '            });',
            '            ',
            '            // Deactivate all tabs',
            '            document.querySelectorAll(".tab").forEach(tab => {',
            '                tab.classList.remove("active");',
            '            });',
            '            ',
            '            // Activate the selected tab',
            '            document.getElementById(tabId).classList.add("active");',
            '            document.querySelector(`.tab[onclick="switchTab(\'${tabId}\')"]`).classList.add("active");',
            '        }',
            '        ',
            '        // Set the corrected and original code',
            '        var correctedCodeElement = document.getElementById("corrected-code-display");',
            '        var originalCodeElement = document.getElementById("original-code-display");',
            '        ',
            '        // This will be replaced with the actual corrected HTML',
            '        correctedCodeElement.textContent = `CORRECTED_HTML_PLACEHOLDER`;',
            '        ',
            '        // This will be replaced with the original HTML',
            '        originalCodeElement.textContent = `ORIGINAL_HTML_PLACEHOLDER`;',
            '    </script>',
            '</body>',
            '</html>'
        ])

    def get_corrected_html(self):
        """Return the corrected HTML as a string"""
        return '\n'.join(self.corrected_html)

    def get_raw_corrected_html(self):
        """Return just the raw corrected HTML without the report structure"""
        return '\n'.join(self.raw_corrected_html)

def process_html(html_code, output_filename="compiled_html_output.html"):
    """Process HTML code and generate output file"""
    start_time = time.time()

    logging.info(f"Starting HTML compilation process")
    logging.info(f"HTML size: {len(html_code)} characters")

    # Create the parser instance
    parser = HTMLCompiler()

    try:

        parser.feed(html_code)


        parser.error_check()


        corrected_html = parser.get_corrected_html()
        raw_corrected_html = parser.get_raw_corrected_html()

        # Replace placeholders with actual HTML (properly escaped)
        escaped_original = html_code.replace('\\', '\\\\').replace('`', '\\`').replace('$', '\\$')
        escaped_corrected = raw_corrected_html.replace('\\', '\\\\').replace('`', '\\`').replace('$', '\\$')

        corrected_html = corrected_html.replace('CORRECTED_HTML_PLACEHOLDER', escaped_corrected)
        corrected_html = corrected_html.replace('ORIGINAL_HTML_PLACEHOLDER', escaped_original)


        file_path = os.path.abspath(output_filename)


        raw_file_path = os.path.abspath("corrected_" + output_filename)
        with open(raw_file_path, "w", encoding="utf-8") as f:
            f.write(raw_corrected_html)


        with open(file_path, "w", encoding="utf-8") as f:
            f.write(corrected_html)

        processing_time = time.time() - start_time

        logging.info(f"HTML compilation completed in {processing_time:.4f} seconds")
        logging.info(f"Output report saved to: {file_path}")
        logging.info(f"Raw corrected HTML saved to: {raw_file_path}")

        # Try to open the file in the default browser
        try:
            webbrowser.open_new(f"file://{file_path}")
            print(f"Compiled HTML saved and opened in browser: {file_path}")
            print(f"Raw corrected HTML saved to: {raw_file_path}")
        except Exception as e:
            logging.error(f"Failed to open browser: {str(e)}")
            print(f"Compiled HTML saved to: {file_path}")
            print(f"Raw corrected HTML saved to: {raw_file_path}")

        return {
            "success": True,
            "file_path": file_path,
            "raw_file_path": raw_file_path,
            "errors": len(parser.errors),
            "warnings": len(parser.warnings),
            "processing_time": processing_time
        }

    except Exception as e:
        logging.error(f"Error during HTML compilation: {str(e)}")
        print(f"Error during compilation: {str(e)}")
        return {
            "success": False,
            "error": str(e)
        }

if __name__ == "__main__":

    html_code = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Test Page</title>
        <meta name="description" content="A test page for HTML compiler">
    </head>
    <body>
        <h1 style="color: blue; font-size: 24px;">Welcome to My Website</h1>
        <p>This is a <b>bold</b> and <i>italic</i> text with some  extra  spaces.</p>
        <div id="main-content">
            <p>List of items:
            <ul>
                <li>Item 1
                <li>Item 2</li>
                <div>Incorrect child of ul</div>
            </ul>
            <img alt="">
            <a>Broken Link</a>
            <a href="https://example.com">Valid Link</a>
            <div id="duplicate">First div with this ID</div>
            <span id="duplicate">Should have unique ID</span>
            <!-- This is a comment -->
            <table>
                <tr>
                    <td>Cell 1</td>
                    <td>Cell 2

            </table>
        </div>
    """

    # Uncomment to get user input for HTML code
    # html_code = input("Enter your HTML code (or press Enter to use the sample): ").strip()
    # if not html_code:
    #     html_code = sample_html

    # Process the HTML
    result = process_html(html_code)

    # Print additional information
    if result["success"]:
        print(f"Found {result['errors']} errors and {result['warnings']} warnings")
        print(f"Processing time: {result['processing_time']:.2f} seconds")

Compiled HTML saved and opened in browser: /content/compiled_html_output.html
Raw corrected HTML saved to: /content/corrected_compiled_html_output.html
Found 15 errors and 2 warnings
Processing time: 0.00 seconds
